# I. Preprocessing Step

In [ ]:
import pandas as pd

reviews = []
labels = []

with open("reviews.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        review, label = line.rsplit(maxsplit=1)
        reviews.append(review)
        labels.append(int(label))

df = pd.DataFrame({
    "review": reviews,
    "sentiment": labels
})

In [ ]:
chat_words = {}

with open("slang.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip().split('=')
        chat_words[line[0]] = line[1]

In [ ]:
df.head()

,review,sentiment
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [ ]:
# lowercasing (to have case insensitive words)
df['review'] = df['review'].str.lower()

In [ ]:
import re
# removing HTML tags
# substituting pattern with ''
def remove_html_tags(text):
  pattern = re.compile('<.*?>')
  return pattern.sub(r'',text)

In [ ]:
df['review'] = df['review'].apply(remove_html_tags)

In [ ]:
# removes simple urls, change pattern for complex urls
def remove_url(text):
  pattern = re.compile(r'https?://\S+|www\.\S+')
  return pattern.sub(r'',text)

In [ ]:
df['review'] = df['review'].apply(remove_url)

In [ ]:
# remove punctuation
# important preprocessing before creating tokens
import string
exclude = string.punctuation

def remove_punc(text):
  for char in exclude:
    text = text.replace(char,'')
  return text

In [ ]:
# faster approach to remove punctuation
def remove_punc_fast(text):
  return text.translate(str.maketrans('','',exclude))

In [ ]:
df['review'] = df['review'].apply(remove_punc)

In [ ]:
# to expand chat abbreviations
def chat_conversion(text):
  new_text = []
  for w in text.split():
    if w.upper() in chat_words:
      new_text.append(remove_punc(chat_words[w.upper()]))
    else:
      new_text.append(w)
  return ' '.join(new_text)

In [ ]:
df['review'] = df['review'].apply(chat_conversion)

In [ ]:
# Spelling correction
# to avoid different tokens for same word
from textblob import TextBlob

def spell_correc(text):

  textblb = TextBlob(text)

  return textblb.correct().string

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# removing stop words
from nltk.corpus import stopwords
stopwds = stopwords.words('english')

def remove_stopwords(text):
  new_text = []

  for w in text.split():
    if w in stopwds:
      continue
    else:
      new_text.append(w)

  return ' '.join(new_text)

In [ ]:
df['review'] = df['review'].apply(remove_stopwords)

In [ ]:
# remove emoji if exists
def remove_emoji(text):
  emoji_pattern = re.compile("["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Misc Symbols and Pictographs
    "\U0001F680-\U0001F6FF"  # Transport and Map Symbols
    "\U0001F700-\U0001F77F"  # Alchemical Symbols
    "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
    "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
    "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
    "\U0001FA00-\U0001FA6F"  # Chess Symbols
    "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
    "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r'', text)

In [ ]:
# demojize
# convert emoji to words
! python -m pip install emoji --upgrade
import emoji

def demojize(text):
  return emoji.demojize(text)

In [ ]:
# Tokenization
# word tokenization and sentence tokenization

# problem in handling
# prefix $ ( " ? e.g. -> (20$)
# suffix km ) , . ! e.g. -> (50km)
# infix . @ - -- / ... e.g. -> (Ph.D f@gmail.com)
# exception let's U.S. (account while punctuation removal)

# can form bigrams and trigram using gensim
# to account words like new delhi, new york city

# Using split function

In [ ]:
# does not handle problems discussed
# can only split based on one char
def split_tokens(text):
  return text.split()

# Use regular expression

In [ ]:
# does not handle problems discussed
# can split based on multiple chars
def reg_tokens(text):
  return re.compile('[.!?]').split(text) # for sentence tokenization

# Use libraries

In [ ]:
# internally handles most problems discussed above not all
from nltk.tokenize import word_tokenize,sent_tokenize
nltk.download('punkt_tab')

def nltk_tokenize(text):
  return word_tokenize(text)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# internally handles most problems discussed above not all
import spacy
nlp = spacy.load('en_core_web.sm')

def spacy_tokens(text):
  return nlp(text)

In [ ]:
# stemming
# to convert inflected word to root word e.g. undoable root word is do
# problem root words may not be a valid word in the language

# choose stemmer based on language
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

# use before tokenization or after tokenization
def stem_words(text):
  return " ".join([ps.stem(word) for word in text.split()]) # use before tokenization


In [ ]:
# lemmetization (slower compared to stemming, so use only if output needs to be shown)
# solves valid word problem while stemming

from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

wordnet_lemmatizer = WordNetLemmatizer()

# use before tokenization or after tokenization
# remember to specify POS for lemmatization
def lemmatize_words(text):
  return " ".join([wordnet_lemmatizer.lemmatize(word,pos='v') for word in text.split()]) # use before tokenization

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
df['review'] = df['review'].apply(lemmatize_words)

In [ ]:
df['review'] = df['review'].apply(nltk_tokenize)
tokenized_reviews = list(df['review'].values.tolist())
print(tokenized_reviews)

[['way', 'plug', 'us', 'unless', 'go', 'converter'], ['good', 'case', 'excellent', 'value'], ['great', 'jawbone'], ['tie', 'charger', 'conversations', 'last', '45', 'minutesmajor', 'problems'], ['mic', 'great'], ['jiggle', 'plug', 'get', 'line', 'right', 'get', 'decent', 'volume'], ['several', 'dozen', 'several', 'hundred', 'contact', 'imagine', 'fun', 'send', 'one', 'one'], ['razr', 'owneryou', 'must'], ['needless', 'say', 'waste', 'money'], ['waste', 'money', 'Tears', 'In', 'My', 'Eyes'], ['sound', 'quality', 'great'], ['impress', 'go', 'original', 'battery', 'extend', 'battery'], ['two', 'seperated', 'mere', '5', 'ft', 'start', 'notice', 'excessive', 'static', 'garble', 'sound', 'headset'], ['good', 'quality', 'though'], ['design', 'odd', 'ear', 'clip', 'comfortable'], ['highly', 'recommend', 'one', 'blue', 'tooth', 'phone'], ['advise', 'everyone', 'fool'], ['far', 'good'], ['work', 'great'], ['click', 'place', 'way', 'make', 'wonder', 'long', 'mechanism', 'would', 'last'], ['go', '

# II. Analysis and Modelling

# Word Cloud

In [ ]:
from wordcloud import WordCloud

concat_reviews = ','.join(list(df['review'].values))

wordcld = WordCloud(background_color="black", max_words=5000, contour_width=3, contour_color='steelblue'
, width=1000, height=800)

wordcld.generate(concat_reviews)

wordcld.to_image()

In [ ]:
!pip install gensim pyLDAvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 70.1 MB/s eta 0:00:00


In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models
import gensim
from gensim.utils import simple_preprocess
import gensim.corpora as corpora

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
id2word = corpora.Dictionary(tokenized_reviews)

texts = tokenized_reviews

corpus = [id2word.doc2bow(text) for text in texts]

print(corpus[0])

In [ ]:
# no. of topics
k = 10
# for large dataset use LdaMulticore which splits document chunks across multiple CPU cores simultaneously
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=k,
    random_state=42,  # Ensures reproducible results
    passes=10,  # Number of training passes over the corpus
    alpha="auto",  # Automatic prior estimation, uses asymmetric Dirichlet updates
    # Updates alpha automatically
    # By default, LDA uses fixed symmetric priors (e.g., a fixed value like alpha=0.01)
    # A symmetric prior treats every single topic equally this assumption is rarely true
)

# # Rule of thumb: workers = available CPU cores - 1
# cores = max(1, os.cpu_count() - 1)

# lda_multicore = LdaMulticore(
#     corpus=corpus,
#     id2word=dictionary,
#     num_topics=10,
#     workers=cores,  # Pass the number of CPU worker processes
#     passes=15,
#     chunksize=2000,
#     random_state=42,
# )

In [ ]:
pyLDAvis.enable_notebook()  # Enable if running in Jupyter / Colab
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word)

In [ ]:
# use bigrams, trigrams and TF IDF removal if word quality is poor
bigram_phrases = gensim.models.Phrases(tokenized_reviews, min_count=5, threshold=50)
trigram_phrases = gensim.models.Phrases(bigram_phrases[tokenized_reviews], threshold=50)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
  return [bigram[doc] for doc in texts]

def make_trigrams(texts):
  return [trigram[bigram[doc]] for doc in texts]

data_bigrams_trigrams = make_trigrams(tokenized_reviews)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# TF IDF removal
from gensim.models import TfidfModel

id2word_new = corpora.Dictionary(data_bigrams_trigrams)

texts_new = data_bigrams_trigrams

corpus_new = [id2word_new.doc2bow(doc) for doc in texts_new]

tfidf = TfidfModel(corpus_new, id2word=id2word_new)

low_value = 0.01

words = []
words_missing_in_tfidf = []
for i in range(0, len(corpus_new)):
    bow = corpus_new[i]
    low_value_words = [] #reinitialize to be safe. You can skip this.
    tfidf_ids = [id for id, value in tfidf[bow]]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words + words_missing_in_tfidf

    for item in drops:
      words.append(id2word_new[item])

    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # The words with tf-idf socre 0 will be missing

    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]

    #reassign
    corpus_new[i] = new_bow

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
k = 10

lda_model = gensim.models.LdaModel(
    corpus=corpus_new,
    id2word=id2word_new,
    num_topics=k,
    random_state=42,
    passes=10,
    alpha="auto",
    eta='auto',
)

In [ ]:
pyLDAvis.enable_notebook()  # Enable if running in Jupyter / Colab
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus_new, id2word_new)

vis

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
9      0.066559  0.086229       1        1  12.526496
0      0.068242 -0.075187       2        1  11.764272
2      0.089580 -0.022523       3        1  11.069019
6     -0.061713  0.039488       4        1  10.648439
8     -0.148941 -0.091058       5        1  10.423390
4      0.050674 -0.101216       6        1   9.503102
5      0.050353 -0.000084       7        1   9.201547
1     -0.074357  0.023159       8        1   8.711470
7     -0.020312  0.110638       9        1   8.257981
3     -0.020084  0.030553      10        1   7.894283, topic_info=        Term       Freq      Total Category  logprob  loglift
135  product  43.000000  43.000000  Default  30.0000  30.0000
10     great  75.000000  75.000000  Default  29.0000  29.0000
73      work  87.000000  87.000000  Default  28.0000  28.0000
176    price  26.000000  26.000000  Default  27.0000  27.0000
134     like  26.000000  26.000000  Default  26.0000  26.0000
..       ...        ...        ...      ...      ...      ...
186     call   2.308031  19.210451  Topic10  -5.2111   0.4200
134     like   2.378560  26.308039  Topic10  -5.1810   0.1357
31       one   2.308041  31.834972  Topic10  -5.2111  -0.0851
92       use   2.343002  53.816942  Topic10  -5.1961  -0.5951
42   quality   2.321032  38.811852  Topic10  -5.2055  -0.2777

[699 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
569       3  0.318606          12
569       5  0.318606          12
214       6  0.654949          15
1284      5  0.643241          18
288       2  0.233836           2
...     ...       ...         ...
971       8  0.875400       wrong
618       8  0.661888        year
618      10  0.330944        year
1449      6  0.654949  yearsgreat
1049      5  0.643242        zero

[1318 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[10, 1, 3, 7, 9, 5, 6, 2, 8, 4])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# III. BERTopic for Topic Modelling

In [ ]:
# use preprocessed sentence without tokenization
texts = df['review'].tolist()
texts

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
# can reduce dimensionality using PCA
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
clusters = clusterer.fit_predict(embeddings)

In [ ]:
# visualize with UMAP
import umap.umap_ as umap
umap_model = umap.UMAP(n_neighbors=15, n_components=2, metric='cosine')
umap_embeddings = umap_model.fit_transform(embeddings)

plt.figure(figsize=(15,7))
plt.scatter(umap_embeddings=[:,0], umap_embeddings[:,1], c=clusters, cmap='tab10')
plt.colorbar()
plt.title("Topic Clusters")
plt.show()

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.0 MB/s eta 0:00:00


In [ ]:
from bertopic import BERTopic

topic_model = BERTopic()
topics, probs = topic_model.fit_transform(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
topic_model.visualize_topics()